In [69]:
import numpy as np
import pandas as pd
import os

H = 63

MODEL_ORDER = ["EWMA", "DCC", "HAR", "SVR", "XGB", "CAB"]

PROXY_DIR = "../proxy"
RESULTS_DIR = "../results"
MODELS_DIR = "../models"

In [70]:
def load_cov_csv(path):
    return pd.read_csv(
        path,
        parse_dates=["Date"]
    ).set_index("Date").sort_index()


# -------------------------
# Load proxy
# -------------------------

proxy_path = os.path.join(PROXY_DIR, f"realized_cov_h{H}.csv")

proxy = load_cov_csv(proxy_path)

print(f"Loaded proxy (h={H})")
print(f"File: {proxy_path}")
print("Shape:", proxy.shape)
print("Date range:", proxy.index.min().date(), "->", proxy.index.max().date())
print()


# -------------------------
# Load models
# -------------------------

models = {}

for model in MODEL_ORDER:

    if model == "EWMA":
        path = os.path.join(
            MODELS_DIR,
            "ewma_cov_lambda094.csv"
        )
    else:
        path = os.path.join(
            RESULTS_DIR,
            f"{model.lower()}_cov_forecast_h{H}.csv"
        )

    if os.path.exists(path):

        models[model] = load_cov_csv(path)

        df = models[model]

        print(f"Loaded {model}")
        print(f"File: {path}")
        print("Shape:", df.shape)
        print("Date range:", df.index.min().date(), "->", df.index.max().date())
        print()

    else:
        print(f"[skip] Missing file for {model}: {path}")

Loaded proxy (h=63)
File: ../proxy/realized_cov_h63.csv
Shape: (2142, 64)
Date range: 2017-03-28 -> 2025-10-02

Loaded EWMA
File: ../models/ewma_cov_lambda094.csv
Shape: (1255, 64)
Date range: 2021-01-04 -> 2025-12-31

Loaded DCC
File: ../results/dcc_cov_forecast_h63.csv
Shape: (1255, 64)
Date range: 2021-01-04 -> 2025-12-31

Loaded HAR
File: ../results/har_cov_forecast_h63.csv
Shape: (1193, 64)
Date range: 2021-01-04 -> 2025-10-02

Loaded SVR
File: ../results/svr_cov_forecast_h63.csv
Shape: (1193, 64)
Date range: 2021-01-04 -> 2025-10-02

Loaded XGB
File: ../results/xgb_cov_forecast_h63.csv
Shape: (1193, 64)
Date range: 2021-01-04 -> 2025-10-02

Loaded CAB
File: ../results/cab_cov_forecast_h63.csv
Shape: (1193, 64)
Date range: 2021-01-04 -> 2025-10-02



In [71]:
# -------------------------
# Align evaluation sample
# -------------------------

common_index = proxy.index

for df in models.values():
    common_index = common_index.intersection(df.index)

proxy = proxy.loc[common_index]

for name in models:
    models[name] = models[name].loc[common_index]


print("Aligned evaluation sample")
print("Start:", common_index.min().date())
print("End  :", common_index.max().date())
print("Observations:", len(common_index))
print()

print("Proxy", proxy.shape)

for name, df in models.items():
    print(name, df.shape)

Aligned evaluation sample
Start: 2021-01-04
End  : 2025-10-02
Observations: 1193

Proxy (1193, 64)
EWMA (1193, 64)
DCC (1193, 64)
HAR (1193, 64)
SVR (1193, 64)
XGB (1193, 64)
CAB (1193, 64)


In [72]:
n_assets = int(np.sqrt(proxy.shape[1]))
print("Number of assets:", n_assets)

Number of assets: 8


In [73]:
# ============================================================
# Matrix utilities
# ============================================================

def row_to_matrix(row, n_assets):
    mat = row.to_numpy(dtype=float).reshape(n_assets, n_assets)
    mat = 0.5 * (mat + mat.T)
    return mat


def make_spd(mat, eps=1e-8):
    mat = 0.5 * (mat + mat.T)

    eigvals, eigvecs = np.linalg.eigh(mat)
    eigvals[eigvals < eps] = eps

    mat_spd = eigvecs @ np.diag(eigvals) @ eigvecs.T
    mat_spd = 0.5 * (mat_spd + mat_spd.T)

    return mat_spd


# ============================================================
# Covariance loss functions
# ============================================================

def qlike_loss(proxy_mat, forecast_mat):

    S = proxy_mat
    H = forecast_mat

    n = S.shape[0]

    H_inv = np.linalg.inv(H)
    trace_term = np.trace(S @ H_inv)

    sign_s, logdet_s = np.linalg.slogdet(S)
    sign_h, logdet_h = np.linalg.slogdet(H)

    if sign_s <= 0 or sign_h <= 0:
        raise ValueError("Non-positive determinant encountered in QLIKE calculation.")

    return float(trace_term - (logdet_s - logdet_h) - n)


def frobenius_loss(proxy_mat, forecast_mat):

    diff = forecast_mat - proxy_mat
    return float(np.sum(diff ** 2))


# ============================================================
# Covariance evaluation
# ============================================================

def evaluate_against_proxy(forecast_df, proxy_df, model_name, n_assets):

    common_dates = forecast_df.index.intersection(proxy_df.index)

    forecast_df = forecast_df.loc[common_dates]
    proxy_df = proxy_df.loc[common_dates]

    records = []
    spd_fix_count = 0

    for dt in common_dates:

        H_mat = row_to_matrix(forecast_df.loc[dt], n_assets)
        S_mat = row_to_matrix(proxy_df.loc[dt], n_assets)

        try:
            qlike = qlike_loss(S_mat, H_mat)

        except ValueError:
            H_mat = make_spd(H_mat)
            qlike = qlike_loss(S_mat, H_mat)
            spd_fix_count += 1

        records.append({
            "Date": dt,
            "QLIKE": qlike,
            "FROBENIUS": frobenius_loss(S_mat, H_mat)
        })

    losses = pd.DataFrame(records).set_index("Date")

    summary = pd.Series({
        "model": model_name,
        "n_obs": len(losses),
        "start_date": losses.index.min().date(),
        "end_date": losses.index.max().date(),
        "avg_qlike": losses["QLIKE"].mean(),
        "avg_frobenius": losses["FROBENIUS"].mean(),
        "spd_fixes": spd_fix_count
    })

    return losses, summary


# ============================================================
# Variance / Correlation utilities
# ============================================================

def covariance_to_variance_vector(cov_mat):
    return np.diag(cov_mat)


def covariance_to_correlation_matrix(cov_mat):

    cov_mat = 0.5 * (cov_mat + cov_mat.T)

    var = np.diag(cov_mat)
    std = np.sqrt(np.maximum(var, 1e-12))

    corr = cov_mat / np.outer(std, std)

    corr = 0.5 * (corr + corr.T)
    np.fill_diagonal(corr, 1.0)

    return corr


# ============================================================
# Component loss functions
# ============================================================

def variance_mse_loss(proxy_mat, forecast_mat):

    var_proxy = covariance_to_variance_vector(proxy_mat)
    var_forecast = covariance_to_variance_vector(forecast_mat)

    return float(np.mean((var_forecast - var_proxy) ** 2))


def correlation_mse_loss(proxy_mat, forecast_mat):

    corr_proxy = covariance_to_correlation_matrix(proxy_mat)
    corr_forecast = covariance_to_correlation_matrix(forecast_mat)

    n = corr_proxy.shape[0]

    tri_i, tri_j = np.triu_indices(n, k=1)

    diff = corr_forecast[tri_i, tri_j] - corr_proxy[tri_i, tri_j]

    return float(np.mean(diff ** 2))


# ============================================================
# Component evaluation
# ============================================================

def evaluate_components_against_proxy(forecast_df, proxy_df, model_name, n_assets):

    common_dates = forecast_df.index.intersection(proxy_df.index)

    forecast_df = forecast_df.loc[common_dates]
    proxy_df = proxy_df.loc[common_dates]

    records = []

    for dt in common_dates:

        H_mat = row_to_matrix(forecast_df.loc[dt], n_assets)
        S_mat = row_to_matrix(proxy_df.loc[dt], n_assets)

        records.append({
            "Date": dt,
            "VAR_MSE": variance_mse_loss(S_mat, H_mat),
            "CORR_MSE": correlation_mse_loss(S_mat, H_mat)
        })

    losses = pd.DataFrame(records).set_index("Date")

    summary = pd.Series({
        "model": model_name,
        "n_obs": len(losses),
        "start_date": losses.index.min().date(),
        "end_date": losses.index.max().date(),
        "avg_var_mse": losses["VAR_MSE"].mean(),
        "avg_corr_mse": losses["CORR_MSE"].mean()
    })

    return losses, summary


# ============================================================
# System volatility
# ============================================================

def system_volatility(df, n_assets):

    vol = df.apply(
        lambda row: np.sqrt(np.trace(row_to_matrix(row, n_assets))),
        axis=1
    )

    return vol

In [74]:
for model_name in MODEL_ORDER:
    if model_name not in models:
        continue

    forecast_df = models[model_name]

    for dt in forecast_df.index.intersection(proxy.index):
        H_mat = row_to_matrix(forecast_df.loc[dt], n_assets)
        sign_h, logdet_h = np.linalg.slogdet(H_mat)

        if sign_h <= 0:
            print(f"{model_name} has non-positive determinant on {dt.date()}")
            break

HAR has non-positive determinant on 2024-04-17


Analysis whole sample 

In [75]:
cov_losses = {}
cov_summaries = []

for model_name in MODEL_ORDER:

    if model_name not in models:
        continue

    forecast_df = models[model_name]

    losses, summary = evaluate_against_proxy(
        forecast_df=forecast_df,
        proxy_df=proxy,
        model_name=model_name,
        n_assets=n_assets
    )

    cov_losses[model_name] = losses
    cov_summaries.append(summary)

cov_summary_table = pd.DataFrame(cov_summaries).set_index("model")
cov_summary_table = cov_summary_table.loc[[m for m in MODEL_ORDER if m in cov_summary_table.index]]

print("Covariance Forecast Performance (Full Sample)")
cov_summary_table

Covariance Forecast Performance (Full Sample)


,n_obs,start_date,end_date,avg_qlike,avg_frobenius,spd_fixes
model,,,,,,
EWMA,1193,2021-01-04,2025-10-02,4.027345183,0.000001095,0
DCC,1193,2021-01-04,2025-10-02,2.535822238,0.000005122,0
HAR,1193,2021-01-04,2025-10-02,28.222285056,0.000001063,43
SVR,1193,2021-01-04,2025-10-02,3.433908238,0.000000608,0
XGB,1193,2021-01-04,2025-10-02,3.337306888,0.000000835,0
CAB,1193,2021-01-04,2025-10-02,18.668952894,0.000000692,0


In [76]:
component_losses = {}
component_summaries = []

for model_name in MODEL_ORDER:

    if model_name not in models:
        continue

    forecast_df = models[model_name]

    losses, summary = evaluate_components_against_proxy(
        forecast_df=forecast_df,
        proxy_df=proxy,
        model_name=model_name,
        n_assets=n_assets
    )

    component_losses[model_name] = losses
    component_summaries.append(summary)

component_summary_table = pd.DataFrame(component_summaries).set_index("model")
component_summary_table = component_summary_table.loc[[m for m in MODEL_ORDER if m in component_summary_table.index]]

print("Variance and Correlation Forecast Performance (Full Sample)")
component_summary_table

Variance and Correlation Forecast Performance (Full Sample)


,n_obs,start_date,end_date,avg_var_mse,avg_corr_mse
model,,,,,
EWMA,1193,2021-01-04,2025-10-02,0.000000105,0.063735925
DCC,1193,2021-01-04,2025-10-02,0.000000613,0.040612964
HAR,1193,2021-01-04,2025-10-02,0.000000096,237.631691273
SVR,1193,2021-01-04,2025-10-02,0.000000051,0.047658693
XGB,1193,2021-01-04,2025-10-02,0.000000068,0.049844467
CAB,1193,2021-01-04,2025-10-02,0.000000061,0.053753012


In [77]:
qlike_compare = pd.DataFrame({
    model_name: cov_losses[model_name]["QLIKE"]
    for model_name in cov_losses
})

frobenius_compare = pd.DataFrame({
    model_name: cov_losses[model_name]["FROBENIUS"]
    for model_name in cov_losses
})

print("Best model by QLIKE share:")
print(qlike_compare.idxmin(axis=1).value_counts(normalize=True))

print("Best model by frobenius share:")
print(frobenius_compare.idxmin(axis=1).value_counts(normalize=True))

Best model by QLIKE share:
DCC    0.427493713
SVR    0.343671417
XGB    0.104777871
HAR    0.059513831
EWMA   0.056999162
CAB    0.007544007
Name: proportion, dtype: float64
Best model by frobenius share:
SVR    0.295892707
CAB    0.208717519
EWMA   0.187761945
HAR    0.139145013
XGB    0.109807209
DCC    0.058675608
Name: proportion, dtype: float64


Analysis regime

In [78]:
proxy_vol = proxy.apply(
    lambda row: np.sqrt(np.trace(row_to_matrix(row, n_assets))),
    axis=1
)

proxy_vol.name = "system_volatility"

low_threshold = proxy_vol.quantile(0.25)
high_threshold = proxy_vol.quantile(0.75)

print("Low volatility threshold:", low_threshold)
print("High volatility threshold:", high_threshold)

vol_regime = pd.Series(index=proxy_vol.index, dtype="object")
vol_regime[proxy_vol <= low_threshold] = "low"
vol_regime[proxy_vol >= high_threshold] = "high"
vol_regime[(proxy_vol > low_threshold) & (proxy_vol < high_threshold)] = "mid"

vol_regime.value_counts()

Low volatility threshold: 0.03634438550066854
High volatility threshold: 0.04709001124194135


mid     595
high    299
low     299
Name: count, dtype: int64

In [79]:
for model in cov_losses:
    cov_losses[model]["regime"] = vol_regime.loc[cov_losses[model].index]

for model in component_losses:
    component_losses[model]["regime"] = vol_regime.loc[component_losses[model].index]

In [80]:
qlike_regime_table = pd.DataFrame({
    model: cov_losses[model].groupby("regime")["QLIKE"].mean()
    for model in MODEL_ORDER if model in cov_losses
})

frob_regime_table = pd.DataFrame({
    model: cov_losses[model].groupby("regime")["FROBENIUS"].mean()
    for model in MODEL_ORDER if model in cov_losses
})

# fast regime rekkefølge
regime_order = ["low", "mid", "high"]

qlike_regime_table = qlike_regime_table.loc[regime_order]
frob_regime_table = frob_regime_table.loc[regime_order]

print("Average QLIKE in regimes (lower is better)")
display(qlike_regime_table)

print("\nAverage Frobenius loss in regimes (lower is better)")
display(frob_regime_table)

Average QLIKE in regimes (lower is better)


,EWMA,DCC,HAR,SVR,XGB,CAB
regime,,,,,,
low,3.056410681,2.095142211,7.618790618,2.403808790,2.211175048,22.843541433
mid,4.505951343,2.600453276,48.376713004,2.787174627,3.124672071,15.938655122
high,4.045869434,2.847888661,8.719141940,5.750985944,4.886574904,19.927565606



Average Frobenius loss in regimes (lower is better)


,EWMA,DCC,HAR,SVR,XGB,CAB
regime,,,,,,
low,0.000000862,0.000001351,0.000000396,0.000000431,0.000000575,0.000000392
mid,0.000000788,0.000003719,0.000000595,0.000000205,0.000000567,0.000000636
high,0.000001941,0.000011685,0.000002663,0.000001585,0.000001627,0.000001103


In [81]:
var_regime_table = pd.DataFrame({
    model: component_losses[model].groupby("regime")["VAR_MSE"].mean()
    for model in MODEL_ORDER if model in component_losses
})

corr_regime_table = pd.DataFrame({
    model: component_losses[model].groupby("regime")["CORR_MSE"].mean()
    for model in MODEL_ORDER if model in component_losses
})

var_regime_table = var_regime_table.loc[regime_order]
corr_regime_table = corr_regime_table.loc[regime_order]

print("Variance MSE by volatility regime (lower is better)")
display(var_regime_table)

print("\nCorrelation MSE by volatility regime (lower is better)")
display(corr_regime_table)

Variance MSE by volatility regime (lower is better)


,EWMA,DCC,HAR,SVR,XGB,CAB
regime,,,,,,
low,0.000000070,0.000000153,0.000000026,0.000000044,0.000000052,0.000000032
mid,0.000000077,0.000000444,0.000000052,0.000000012,0.000000044,0.000000058
high,0.000000197,0.000001409,0.000000255,0.000000134,0.000000133,0.000000097



Correlation MSE by volatility regime (lower is better)


,EWMA,DCC,HAR,SVR,XGB,CAB
regime,,,,,,
low,0.062991384,0.029806332,0.054976631,0.040065604,0.036125196,0.046761204
mid,0.066831133,0.039022906,476.351087426,0.042438859,0.053011833,0.050551662
high,0.058321108,0.054583756,0.164791495,0.065639077,0.057260785,0.067115400


In [82]:
realized_vol = system_volatility(proxy, n_assets)

model_vol = {
    model: system_volatility(models[model], n_assets)
    for model in models
}

vol_df = pd.DataFrame({
    "Realized": realized_vol,
    **model_vol
}).dropna()

In [83]:
import plotly.graph_objects as go

fig = go.Figure()

# Realized volatility
fig.add_trace(go.Scatter(
    x=vol_df.index,
    y=vol_df["Realized"],
    mode="lines",
    name="Realized volatility",
    line=dict(width=2)
))

# Model forecasts
for model in models:
    
    fig.add_trace(go.Scatter(
        x=vol_df.index,
        y=vol_df[model],
        mode="lines",
        name=f"{model} forecast",
        line=dict(width=1)
    ))

# Low volatility threshold
fig.add_hline(
    y=low_threshold,
    line_dash="dash",
    line_color="green",
    annotation_text="Low volatility threshold",
    annotation_position="bottom right"
)

# High volatility threshold
fig.add_hline(
    y=high_threshold,
    line_dash="dash",
    line_color="red",
    annotation_text="High volatility threshold",
    annotation_position="top right"
)

fig.update_layout(
    title=f"System Volatility – Horizon h={H}",
    xaxis_title="Date",
    yaxis_title="Volatility",
    hovermode="x unified",
    template="plotly_white"
)

fig.show()

In [84]:
# ============================================================
# Additional visual diagnostics for covariance structure
# Uses existing variables in this notebook:
#   proxy, models, n_assets, row_to_matrix, H
# ============================================================

import numpy as np
import plotly.graph_objects as go

# ------------------------------------------------------------
# 1) Helper: average off-diagonal covariance
# ------------------------------------------------------------
def avg_offdiag_covariance(df, n_assets):
    def _avg_offdiag(row):
        Sigma = row_to_matrix(row, n_assets)
        mask = ~np.eye(n_assets, dtype=bool)
        return Sigma[mask].mean()
    return df.apply(_avg_offdiag, axis=1)

# ------------------------------------------------------------
# 2) Level plot: average off-diagonal covariance
# ------------------------------------------------------------
proxy_avg_cov = avg_offdiag_covariance(proxy, n_assets)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=proxy_avg_cov.index,
        y=proxy_avg_cov.values,
        mode="lines",
        name="Proxy",
        line=dict(width=3)
    )
)

for model_name, df in models.items():
    model_avg_cov = avg_offdiag_covariance(df, n_assets).reindex(proxy_avg_cov.index)

    fig.add_trace(
        go.Scatter(
            x=model_avg_cov.index,
            y=model_avg_cov.values,
            mode="lines",
            name=model_name
        )
    )

fig.update_layout(
    title=f"Average Off-Diagonal Covariance – Horizon h={H}",
    xaxis_title="Date",
    yaxis_title="Average covariance",
    hovermode="x unified",
    template="plotly_white"
)

fig.show()

# ------------------------------------------------------------
# 3) Error plot: model minus proxy
# Positive = overestimation
# Negative = underestimation
# ------------------------------------------------------------
fig = go.Figure()

for model_name, df in models.items():
    model_avg_cov = avg_offdiag_covariance(df, n_assets).reindex(proxy_avg_cov.index)
    error = model_avg_cov - proxy_avg_cov

    fig.add_trace(
        go.Scatter(
            x=error.index,
            y=error.values,
            mode="lines",
            name=model_name
        )
    )

fig.add_hline(y=0, line_dash="dash", line_color="black")

fig.update_layout(
    title=f"Average Off-Diagonal Covariance Error vs Proxy – Horizon h={H}",
    xaxis_title="Date",
    yaxis_title="Model - Proxy",
    hovermode="x unified",
    template="plotly_white"
)

fig.show()

In [85]:
from IPython.display import display

pd.options.display.float_format = '{:.9f}'.format

print("============================================================")
print(f"FINAL EVALUATION SUMMARY (H = {H})")
print("============================================================")

print("\n1. Covariance Forecast Performance (Full Sample)")
display(cov_summary_table)

print("\n2. Variance and Correlation Forecast Performance (Full Sample)")
display(component_summary_table)

print("\n3. Average QLIKE by Volatility Regime")
display(qlike_regime_table)

print("\n4. Average Frobenius Loss by Volatility Regime")
display(frob_regime_table)

print("\n5. Variance MSE by Volatility Regime")
display(var_regime_table)

print("\n6. Correlation MSE by Volatility Regime")
display(corr_regime_table)

FINAL EVALUATION SUMMARY (H = 63)

1. Covariance Forecast Performance (Full Sample)


,n_obs,start_date,end_date,avg_qlike,avg_frobenius,spd_fixes
model,,,,,,
EWMA,1193,2021-01-04,2025-10-02,4.027345183,0.000001095,0
DCC,1193,2021-01-04,2025-10-02,2.535822238,0.000005122,0
HAR,1193,2021-01-04,2025-10-02,28.222285056,0.000001063,43
SVR,1193,2021-01-04,2025-10-02,3.433908238,0.000000608,0
XGB,1193,2021-01-04,2025-10-02,3.337306888,0.000000835,0
CAB,1193,2021-01-04,2025-10-02,18.668952894,0.000000692,0



2. Variance and Correlation Forecast Performance (Full Sample)


,n_obs,start_date,end_date,avg_var_mse,avg_corr_mse
model,,,,,
EWMA,1193,2021-01-04,2025-10-02,0.000000105,0.063735925
DCC,1193,2021-01-04,2025-10-02,0.000000613,0.040612964
HAR,1193,2021-01-04,2025-10-02,0.000000096,237.631691273
SVR,1193,2021-01-04,2025-10-02,0.000000051,0.047658693
XGB,1193,2021-01-04,2025-10-02,0.000000068,0.049844467
CAB,1193,2021-01-04,2025-10-02,0.000000061,0.053753012



3. Average QLIKE by Volatility Regime


...

,EWMA,DCC,HAR,SVR,XGB,CAB
regime,,,,,,
low,0.000000862,0.000001351,0.000000396,0.000000431,0.000000575,0.000000392
mid,0.000000788,0.000003719,0.000000595,0.000000205,0.000000567,0.000000636
high,0.000001941,0.000011685,0.000002663,0.000001585,0.000001627,0.000001103



5. Variance MSE by Volatility Regime


,EWMA,DCC,HAR,SVR,XGB,CAB
regime,,,,,,
low,0.000000070,0.000000153,0.000000026,0.000000044,0.000000052,0.000000032
mid,0.000000077,0.000000444,0.000000052,0.000000012,0.000000044,0.000000058
high,0.000000197,0.000001409,0.000000255,0.000000134,0.000000133,0.000000097



6. Correlation MSE by Volatility Regime


,EWMA,DCC,HAR,SVR,XGB,CAB
regime,,,,,,
low,0.062991384,0.029806332,0.054976631,0.040065604,0.036125196,0.046761204
mid,0.066831133,0.039022906,476.351087426,0.042438859,0.053011833,0.050551662
high,0.058321108,0.054583756,0.164791495,0.065639077,0.057260785,0.067115400
